
### About Dataset

Context: This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content 5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset. Columns -->

• asin - ID of the product, like B000FA64PK

• helpful - helpfulness rating of the review - example: 2/3.

• overall - rating of the product.

• reviewText - text of the review (heading).

• reviewTime - time of the review (raw).

• reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN

• reviewerName - name of the reviewer.

• summary - summary of the review (description).

• unixReviewTime - unix timestamp.

Acknowledgements This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

*Inspiration*

• Sentiment analysis on reviews.

• Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.

• Fake reviews/ outliers.

• Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).

• Any other interesting analysis

### Best Practises

1. Preprocessing And Cleaning
2. Train Test Split
3. BOW,TFIDF,Word2vec
4. Train ML algorithms

In [2]:
# Load the dataset
import pandas as pd
data = pd.read_csv('../data/all_kindle_review.csv')
data.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
df = data[['reviewText','rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [4]:
df.shape

(12000, 2)

In [5]:
## Missing Values
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [6]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [7]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [8]:
## Preprocessing And Cleaning 

In [9]:
# Positive review is 1 and Negative review is 0. So, we will convert the rating column to binary values.
df['rating'] = df['rating'].apply(lambda x: 0 if x<3 else 1)

In [10]:
df['rating'].unique()

array([1, 0])

In [11]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [12]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [13]:
## 1. Lower all the cases
df['reviewText'] = df['reviewText'].str.lower()

In [14]:
df['reviewText'].head()

0    jace rankin may be short, but he's nothing to ...
1    great short read.  i didn't want to put it dow...
2    i'll start by saying this is the first of four...
3    aggie is angela lansbury who carries pocketboo...
4    i did not expect this type of book to be in li...
Name: reviewText, dtype: str

In [15]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Akshat Kumar
[nltk_data]     Singh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
from bs4 import BeautifulSoup

In [18]:
## Removing special characters
df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
## Remove the stopswords
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
## Remove url 
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())
## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

In [19]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [20]:
## Lemmatizer
from nltk.stem import WordNetLemmatizer
lemmatizer=WordNetLemmatizer()

In [21]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [22]:
df['reviewText']=df['reviewText'].apply(lambda x:lemmatize_words(x))

In [23]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [24]:
## Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(df['reviewText'],df['rating'],
                                              test_size=0.20)

In [25]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test).toarray()

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray()

In [27]:
X_train_bow

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(9600, 35894))

In [28]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)

In [29]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [30]:
y_pred_bow=nb_model_bow.predict(X_test_bow)

In [31]:
y_pred_tfidf=nb_model_bow.predict(X_test_tfidf)

In [32]:
confusion_matrix(y_test,y_pred_bow)

array([[531, 235],
       [758, 876]])

In [33]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

BOW accuracy:  0.58625


In [34]:
confusion_matrix(y_test,y_pred_tfidf)

array([[523, 243],
       [746, 888]])

In [35]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.5879166666666666


In [36]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression

train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]

word2vec_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

max_tokens = 50

def padded_word2vec_vectors(tokenized_reviews):
    vectors = np.zeros((len(tokenized_reviews), max_tokens * word2vec_model.vector_size))
    for row_index, tokens in enumerate(tokenized_reviews):
        known_tokens = [token for token in tokens if token in word2vec_model.wv][:max_tokens]
        if known_tokens:
            token_vectors = np.asarray([word2vec_model.wv[token] for token in known_tokens])
            vectors[row_index, :token_vectors.size] = token_vectors.ravel()
    return vectors

X_train_word2vec = padded_word2vec_vectors(train_tokens)
X_test_word2vec = padded_word2vec_vectors(test_tokens)

word2vec_classifier = LogisticRegression(max_iter=1000, random_state=42)
word2vec_classifier.fit(X_train_word2vec, y_train)

print("Word2Vec vocabulary size:", len(word2vec_model.wv))
print("Word2Vec feature shape:", X_train_word2vec.shape)

Word2Vec vocabulary size: 15799
Word2Vec feature shape: (9600, 5000)


In [37]:
y_pred_word2vec = word2vec_classifier.predict(X_test_word2vec)

print("Word2Vec accuracy:", accuracy_score(y_test, y_pred_word2vec))
print("Word2Vec confusion matrix:")
print(confusion_matrix(y_test, y_pred_word2vec))

Word2Vec accuracy: 0.7204166666666667
Word2Vec confusion matrix:
[[ 413  353]
 [ 318 1316]]


In [38]:
def average_word2vec_vectors(tokenized_reviews):
    vectors = np.zeros((len(tokenized_reviews), word2vec_model.vector_size))
    for row_index, tokens in enumerate(tokenized_reviews):
        known_tokens = [token for token in tokens if token in word2vec_model.wv]
        if known_tokens:
            vectors[row_index] = np.mean(
                [word2vec_model.wv[token] for token in known_tokens], axis=0
            )
    return vectors

X_train_avg_word2vec = average_word2vec_vectors(train_tokens)
X_test_avg_word2vec = average_word2vec_vectors(test_tokens)

avg_word2vec_classifier = LogisticRegression(max_iter=1000, random_state=42)
avg_word2vec_classifier.fit(X_train_avg_word2vec, y_train)

print("Average Word2Vec feature shape:", X_train_avg_word2vec.shape)

Average Word2Vec feature shape: (9600, 100)


In [40]:
y_pred_avg_word2vec = avg_word2vec_classifier.predict(X_test_avg_word2vec)

print("Average Word2Vec - Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_avg_word2vec))
print("Classification report:")
print(classification_report(y_test, y_pred_avg_word2vec))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_avg_word2vec))

Average Word2Vec - Logistic Regression
Accuracy: 0.7716666666666666
Classification report:
              precision    recall  f1-score   support

           0       0.69      0.52      0.59       766
           1       0.80      0.89      0.84      1634

    accuracy                           0.77      2400
   macro avg       0.74      0.71      0.72      2400
weighted avg       0.76      0.77      0.76      2400

Confusion matrix:
[[ 401  365]
 [ 183 1451]]


In [41]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [42]:
random_forest_classifier = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)
random_forest_classifier.fit(X_train_avg_word2vec, y_train)
y_pred_random_forest = random_forest_classifier.predict(X_test_avg_word2vec)

print("Average Word2Vec - Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_random_forest))
print("Classification report:")
print(classification_report(y_test, y_pred_random_forest))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_random_forest))

Average Word2Vec - Random Forest
Accuracy: 0.7683333333333333
Classification report:
              precision    recall  f1-score   support

           0       0.67      0.54      0.60       766
           1       0.80      0.88      0.84      1634

    accuracy                           0.77      2400
   macro avg       0.74      0.71      0.72      2400
weighted avg       0.76      0.77      0.76      2400

Confusion matrix:
[[ 412  354]
 [ 202 1432]]


In [43]:
svm_classifier = SVC(kernel="rbf", C=1.0)
svm_classifier.fit(X_train_avg_word2vec, y_train)
y_pred_svm = svm_classifier.predict(X_test_avg_word2vec)

print("Average Word2Vec - SVM")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("Classification report:")
print(classification_report(y_test, y_pred_svm))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_svm))

Average Word2Vec - SVM
Accuracy: 0.76875
Classification report:
              precision    recall  f1-score   support

           0       0.69      0.50      0.58       766
           1       0.79      0.89      0.84      1634

    accuracy                           0.77      2400
   macro avg       0.74      0.70      0.71      2400
weighted avg       0.76      0.77      0.76      2400

Confusion matrix:
[[ 383  383]
 [ 172 1462]]


In [44]:
gaussian_nb_classifier = GaussianNB()
gaussian_nb_classifier.fit(X_train_avg_word2vec, y_train)
y_pred_gaussian_nb = gaussian_nb_classifier.predict(X_test_avg_word2vec)

print("Average Word2Vec - Gaussian Naive Bayes")
print("Accuracy:", accuracy_score(y_test, y_pred_gaussian_nb))
print("Classification report:")
print(classification_report(y_test, y_pred_gaussian_nb))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred_gaussian_nb))

Average Word2Vec - Gaussian Naive Bayes
Accuracy: 0.69
Classification report:
              precision    recall  f1-score   support

           0       0.51      0.75      0.61       766
           1       0.85      0.66      0.74      1634

    accuracy                           0.69      2400
   macro avg       0.68      0.71      0.68      2400
weighted avg       0.74      0.69      0.70      2400

Confusion matrix:
[[ 574  192]
 [ 552 1082]]


In [48]:
import pickle
from pathlib import Path

best_model_bundle = {
    "model": avg_word2vec_classifier,
    "word2vec_model": word2vec_model,
    "embedding_type": "average_word2vec",
    "vector_size": word2vec_model.vector_size,
    "classes": avg_word2vec_classifier.classes_.tolist(),
    "validation_accuracy": accuracy_score(y_test, y_pred_avg_word2vec)
}

model_path = Path("../model/best_sentiment_model.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)

with model_path.open("wb") as file:
    pickle.dump(best_model_bundle, file)

with model_path.open("rb") as file:
    loaded_model_bundle = pickle.load(file)

print(f"Saved model to: {model_path.resolve()}")
print("Saved model type:", type(loaded_model_bundle["model"]).__name__)
print("Validation accuracy:", loaded_model_bundle["validation_accuracy"])
print("Bundle keys:", list(loaded_model_bundle.keys()))

Saved model to: E:\Machine Learning Krish Naik\NLP\Sentiment Analyzer\model\best_sentiment_model.pkl
Saved model type: LogisticRegression
Validation accuracy: 0.7716666666666666
Bundle keys: ['model', 'word2vec_model', 'embedding_type', 'vector_size', 'classes', 'validation_accuracy']


In [49]:
preprocessor_bundle = {
    "lowercase": True,
    "special_characters_pattern": r"[^a-z A-z 0-9-]+",
    "url_pattern": r"(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?",
    "remove_html": True,
    "remove_extra_spaces": True,
    "remove_stopwords": True,
    "stopwords_language": "english",
    "stopwords": sorted(stopwords.words("english")),
    "lemmatize": True,
    "lemmatizer": lemmatizer,
    "embedding_type": "average_word2vec",
    "vector_size": word2vec_model.vector_size
}

preprocessor_path = Path("../model/preprocessor.pkl")
preprocessor_path.parent.mkdir(parents=True, exist_ok=True)

with preprocessor_path.open("wb") as file:
    pickle.dump(preprocessor_bundle, file)

with preprocessor_path.open("rb") as file:
    loaded_preprocessor_bundle = pickle.load(file)

print(f"Saved preprocessor to: {preprocessor_path.resolve()}")
print("Stopword count:", len(loaded_preprocessor_bundle["stopwords"]))
print("Lemmatizer type:", type(loaded_preprocessor_bundle["lemmatizer"]).__name__)
print("Bundle keys:", list(loaded_preprocessor_bundle.keys()))

Saved preprocessor to: E:\Machine Learning Krish Naik\NLP\Sentiment Analyzer\model\preprocessor.pkl
Stopword count: 198
Lemmatizer type: WordNetLemmatizer
Bundle keys: ['lowercase', 'special_characters_pattern', 'url_pattern', 'remove_html', 'remove_extra_spaces', 'remove_stopwords', 'stopwords_language', 'stopwords', 'lemmatize', 'lemmatizer', 'embedding_type', 'vector_size']
